# Notebook 03 — Instruction-Tuned LLM Models
## Member C: LLaVA-NeXT-Video-7B · Video-LLaMA-2

**Run after:** `notebook_00_infrastructure.ipynb`

Saves the same full artifact set as Notebooks 01–02, plus:
- `LLaVA_NeXT_quantisation_ablation.json` — fp16 vs 4-bit NF4 comparison


---
## §1 — Setup

In [ ]:
import sys, os, json, gc, re, textwrap, time
import numpy as np
import torch
import torch.nn as nn
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

sys.path.insert(0, "."); sys.path.insert(0, "shared")

with open("shared/config.json") as f: cfg = json.load(f)
DEVICE=cfg["DEVICE"]; NUM_FRAMES=cfg["NUM_FRAMES"]
MAX_CAPTION_LEN=cfg["MAX_CAPTION_LEN"]; EVAL_SUBSET=cfg["EVAL_SUBSET"]
MSVD_VIDEO_DIR=cfg["MSVD_VIDEO_DIR"]; MSRVTT_VIDEO_DIR=cfg["MSRVTT_VIDEO_DIR"]
RESULTS_DIR=cfg["RESULTS_DIR"]
USE_4BIT=(torch.cuda.is_available() and
          torch.cuda.get_device_properties(0).total_memory/1024**3 < 14.0)

import importlib.util
_spec=importlib.util.spec_from_file_location("utils","shared/utils.py")
_mod=importlib.util.load_module_from_spec(_spec); _spec.loader.exec_module(_mod)
sample_frames=_mod.sample_frames; VideoCaptionDataset=_mod.VideoCaptionDataset
CaptionEvaluator=_mod.CaptionEvaluator; run_evaluation_full=_mod.run_evaluation_full
save_all_artifacts=_mod.save_all_artifacts; run_frame_ablation=_mod.run_frame_ablation
show_scores=_mod.show_scores; show_examples=_mod.show_examples

with open("shared/msvd_captions.json")   as f: msvd_captions  =json.load(f)
with open("shared/msvd_ok_ids.json")     as f: msvd_ok        =json.load(f)
with open("shared/msrvtt_captions.json") as f: msrvtt_captions=json.load(f)
with open("shared/msrvtt_ok_ids.json")   as f: msrvtt_ok      =json.load(f)

msvd_ds  =VideoCaptionDataset(msvd_ok,  MSVD_VIDEO_DIR,  msvd_captions,  NUM_FRAMES)
msrvtt_ds=VideoCaptionDataset(msrvtt_ok,MSRVTT_VIDEO_DIR,msrvtt_captions,NUM_FRAMES)

print(f"DEVICE={DEVICE} USE_4BIT={USE_4BIT}")
print(f"MSVD={len(msvd_ds)} MSR-VTT={len(msrvtt_ds)}")
print("Infrastructure loaded.")


---
## §0 — Experiment Tracking (MLflow)

All evaluation results are logged to MLflow for experiment reproducibility.
View results at `http://localhost:5000` or inside Docker at `http://mlflow:5000`.

> **Rubric Item 7 (10 pts):** Modern Industry Standards — MLOps experiment tracking
> **Rubric Item 6 (10 pts):** Docker containerisation — see `Dockerfile` & `docker-compose.yml`


In [ ]:
# ── MLflow experiment tracking setup ─────────────────────────────────────────
# Connects to the MLflow server (local or Docker service).
# Rubric Item 7 — Modern Industry Standard (MLOps)
import sys, importlib.util
_mspec = importlib.util.spec_from_file_location('mlflow_utils','mlflow_utils.py')
_mmod  = importlib.util.load_module_from_spec(_mspec)
_mspec.loader.exec_module(_mmod)
start_run_ctx     = _mmod.start_run
log_eval_results  = _mmod.log_eval_results
log_ablation      = _mmod.log_ablation
log_figure        = _mmod.log_figure
print_run_url     = _mmod.print_run_url

try:
    import mlflow
    mlflow.set_tracking_uri(_mmod.MLFLOW_URI)
    mlflow.set_experiment(_mmod.EXPERIMENT_NAME)
    print(f'MLflow tracking URI : {_mmod.MLFLOW_URI}')
    print(f'Experiment          : {_mmod.EXPERIMENT_NAME}')
    MLFLOW_ACTIVE = True
except Exception as e:
    print(f'MLflow not available ({e}) — continuing without tracking')
    MLFLOW_ACTIVE = False


In [ ]:
def eval_and_save(model, model_name):
    """Evaluate on both datasets, save ALL artifacts. No reruns needed."""
    for ds, ds_name in [(msvd_ds,"MSVD"),(msrvtt_ds,"MSR-VTT")]:
        print(f"\nEvaluating {model_name} on {ds_name}...")
        result = run_evaluation_full(
            model, ds,
            label=f"{model_name}/{ds_name}",
            max_samples=EVAL_SUBSET,
            compute_per_video=True,
        )
        show_scores(result["scores"], model_name, ds_name)
        show_examples(result["examples"])
        print(f"  mean latency: {result['timing']['mean_s']:.3f}s/video")
        print("  Saving all artifacts...")
        save_all_artifacts(result, model_name, ds_name, RESULTS_DIR)

print("eval_and_save() ready.")


---
## §2 — LLaVA-NeXT-Video-7B

Architecture: `Frames → SigLIP → MLP projector → Mistral-7B (instruction-tuned)`  
Auto-selects fp16 or 4-bit NF4 based on available VRAM

In [ ]:
from transformers import LlavaNextVideoProcessor, LlavaNextVideoForConditionalGeneration

class LLaVANextVideoCaptioner:
    """
    LLaVA-NeXT-Video-7B.
    SigLIP visual encoder + two-layer MLP projector + Mistral-7B.
    Instruction-prompted: natural language drives caption style.
    Supports fp16 and 4-bit NF4 quantisation.
    """
    MODEL_ID="llava-hf/LLaVA-NeXT-Video-7B-hf"
    def __init__(self,device=DEVICE,use_4bit=USE_4BIT):
        self.device=device
        print(f"  Loading {self.MODEL_ID} (4-bit={use_4bit})")
        self.proc=LlavaNextVideoProcessor.from_pretrained(self.MODEL_ID)
        if use_4bit:
            from transformers import BitsAndBytesConfig
            bnb=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type="nf4",
                                   bnb_4bit_compute_dtype=torch.float16,
                                   bnb_4bit_use_double_quant=True)
            self.model=LlavaNextVideoForConditionalGeneration.from_pretrained(
                self.MODEL_ID,quantization_config=bnb,device_map="auto")
        else:
            self.model=LlavaNextVideoForConditionalGeneration.from_pretrained(
                self.MODEL_ID,torch_dtype=torch.float16,device_map="auto")
        self.model.eval()

    @torch.no_grad()
    def __call__(self,frames,max_new=MAX_CAPTION_LEN):
        video_np=np.stack([np.array(f.resize((224,224))) for f in frames])
        conv=[{"role":"user","content":[
            {"type":"video"},
            {"type":"text","text":"Describe what is happening in this video in one concise sentence."}
        ]}]
        prompt=self.proc.apply_chat_template(conv,add_generation_prompt=True)
        inp=self.proc(text=prompt,videos=video_np,
                      return_tensors="pt").to(self.model.device)
        out=self.model.generate(**inp,max_new_tokens=max_new,
                                num_beams=4,no_repeat_ngram_size=2)
        new=out[:,inp["input_ids"].shape[1]:]
        return self.proc.batch_decode(new,skip_special_tokens=True)[0].strip()

print("Loading LLaVA-NeXT-Video-7B..."); llava=LLaVANextVideoCaptioner()
print("  Ready.")


In [ ]:
eval_and_save(llava,"LLaVA-NeXT")
del llava; gc.collect(); torch.cuda.empty_cache() if DEVICE=="cuda" else None


#### Inline results: LLaVA-NeXT

In [ ]:
# ── Inline results chart — LLaVA-NeXT ─────────────────────────────────────
import matplotlib.pyplot as plt, numpy as np

_metrics = ['BLEU_1','BLEU_4','METEOR','CIDEr','ROUGE_L']
_colors  = ['#4C72B0','#C44E52']
_fig, _ax = plt.subplots(figsize=(10, 4))
_x = np.arange(len(_metrics)); _w = 0.35
for _j, (_ds, _sc, _col) in enumerate([
        ('MSVD',    scores_llava_msvd,   '#4C72B0'),
        ('MSR-VTT', scores_llava_msrvtt,  '#C44E52')]):
    _vals = [_sc[m] for m in _metrics]
    _bars = _ax.bar(_x+(_j-0.5)*_w, _vals, _w, label=_ds, color=_col, alpha=0.85)
    for _b,_v in zip(_bars,_vals):
        if _v>0.005: _ax.text(_b.get_x()+_b.get_width()/2, _b.get_height()+0.003,
                              f'{_v:.3f}',ha='center',va='bottom',fontsize=8)
_ax.set_xticks(_x); _ax.set_xticklabels(_metrics, fontsize=10)
_ax.set_ylabel('Score', fontsize=10)
_ax.set_title('LLaVA-NeXT — MSVD vs MSR-VTT', fontsize=12)
_ax.legend(fontsize=10)
_ax.spines[['top','right']].set_visible(False)
_ax.yaxis.grid(True, alpha=0.3, linestyle='--'); _ax.set_axisbelow(True)
plt.tight_layout()
_fig_path = os.path.join(RESULTS_DIR,'llava_inline_chart.png')
plt.savefig(_fig_path, dpi=150, bbox_inches='tight')
plt.show()
if MLFLOW_ACTIVE:
    with start_run_ctx('LLaVA-NeXT','Both'):
        log_eval_results({'scores':scores_llava_msvd,'hypotheses':{},'references':{},'timing':{}},                          'LLaVA-NeXT','MSVD')
        log_eval_results({'scores':scores_llava_msrvtt,'hypotheses':{},'references':{},'timing':{}},                          'LLaVA-NeXT','MSR-VTT')
        log_figure(_fig_path)
    print_run_url()


---
## §3 — Video-LLaMA-2

Architecture: `Frames → CLIP ViT-G → Temporal Q-Former → LLaMA-2-7B`  
Key difference from LLaVA-NeXT: explicit temporal Q-Former vs MLP projection

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoProcessor

class VideoLLaMACaptioner:
    """
    Video-LLaMA-2-7B.
    Temporal Q-Former explicitly models inter-frame dynamics before LLaMA-2.
    Key architectural contrast to LLaVA-NeXT (MLP vs Q-Former bridging).
    """
    MODEL_ID="DAMO-NLP-SG/VideoLLaMA2-7B"
    def __init__(self,device=DEVICE,use_4bit=USE_4BIT):
        self.device=device
        print(f"  Loading {self.MODEL_ID} (4-bit={use_4bit})")
        try:
            from transformers import VideoLlama2ForConditionalGeneration
            Cls=VideoLlama2ForConditionalGeneration
        except ImportError:
            Cls=AutoModelForCausalLM
        dtype=torch.float16 if device=="cuda" else torch.float32
        if use_4bit:
            from transformers import BitsAndBytesConfig
            bnb=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type="nf4",
                                   bnb_4bit_compute_dtype=torch.float16,
                                   bnb_4bit_use_double_quant=True)
            self.model=Cls.from_pretrained(self.MODEL_ID,
                quantization_config=bnb,device_map="auto")
        else:
            self.model=Cls.from_pretrained(self.MODEL_ID,
                torch_dtype=dtype,device_map="auto")
        self.model.eval()
        try:
            self.proc=AutoProcessor.from_pretrained(self.MODEL_ID)
            self.use_proc=True
        except Exception:
            self.tok=AutoTokenizer.from_pretrained(self.MODEL_ID)
            self.tok.pad_token=self.tok.eos_token; self.use_proc=False

    @torch.no_grad()
    def __call__(self,frames,max_new=MAX_CAPTION_LEN):
        video_np=np.stack([np.array(f.resize((224,224))) for f in frames])
        prompt="Describe the video content in one sentence."
        if self.use_proc:
            inp=self.proc(text=prompt,videos=video_np,
                          return_tensors="pt").to(self.model.device)
            out=self.model.generate(**inp,max_new_tokens=max_new,num_beams=4)
            new=out[:,inp["input_ids"].shape[1]:]
            return self.proc.batch_decode(new,skip_special_tokens=True)[0].strip()
        else:
            enc=self.tok(prompt,return_tensors="pt").to(self.model.device)
            out=self.model.generate(**enc,max_new_tokens=max_new,num_beams=4)
            return self.tok.decode(out[0][enc["input_ids"].shape[1]:],
                                   skip_special_tokens=True).strip()

print("Loading Video-LLaMA-2..."); vllama=VideoLLaMACaptioner()
print("  Ready.")


In [ ]:
eval_and_save(vllama,"Video-LLaMA-2")
del vllama; gc.collect(); torch.cuda.empty_cache() if DEVICE=="cuda" else None


#### Inline results: Video-LLaMA-2

In [ ]:
# ── Inline results chart — Video-LLaMA-2 ─────────────────────────────────────
import matplotlib.pyplot as plt, numpy as np

_metrics = ['BLEU_1','BLEU_4','METEOR','CIDEr','ROUGE_L']
_colors  = ['#4C72B0','#C44E52']
_fig, _ax = plt.subplots(figsize=(10, 4))
_x = np.arange(len(_metrics)); _w = 0.35
for _j, (_ds, _sc, _col) in enumerate([
        ('MSVD',    scores_vllama_msvd,   '#4C72B0'),
        ('MSR-VTT', scores_vllama_msrvtt,  '#C44E52')]):
    _vals = [_sc[m] for m in _metrics]
    _bars = _ax.bar(_x+(_j-0.5)*_w, _vals, _w, label=_ds, color=_col, alpha=0.85)
    for _b,_v in zip(_bars,_vals):
        if _v>0.005: _ax.text(_b.get_x()+_b.get_width()/2, _b.get_height()+0.003,
                              f'{_v:.3f}',ha='center',va='bottom',fontsize=8)
_ax.set_xticks(_x); _ax.set_xticklabels(_metrics, fontsize=10)
_ax.set_ylabel('Score', fontsize=10)
_ax.set_title('Video-LLaMA-2 — MSVD vs MSR-VTT', fontsize=12)
_ax.legend(fontsize=10)
_ax.spines[['top','right']].set_visible(False)
_ax.yaxis.grid(True, alpha=0.3, linestyle='--'); _ax.set_axisbelow(True)
plt.tight_layout()
_fig_path = os.path.join(RESULTS_DIR,'vllama_inline_chart.png')
plt.savefig(_fig_path, dpi=150, bbox_inches='tight')
plt.show()
if MLFLOW_ACTIVE:
    with start_run_ctx('Video-LLaMA-2','Both'):
        log_eval_results({'scores':scores_vllama_msvd,'hypotheses':{},'references':{},'timing':{}},                          'Video-LLaMA-2','MSVD')
        log_eval_results({'scores':scores_vllama_msrvtt,'hypotheses':{},'references':{},'timing':{}},                          'Video-LLaMA-2','MSR-VTT')
        log_figure(_fig_path)
    print_run_url()


---
## §4 — Quantisation Ablation (LLaVA-NeXT)

Measures quality-memory trade-off between fp16 and 4-bit NF4.

In [ ]:
quant_abl={}
for use_4bit, q_label in [(False,"fp16"),(True,"4bit_NF4")]:
    if not torch.cuda.is_available(): print(f"Skip {q_label} — no GPU"); continue
    print(f"\nQuantisation ablation: LLaVA-NeXT ({q_label})...")
    m=LLaVANextVideoCaptioner(use_4bit=use_4bit)
    # 100-video subset for speed
    ds_sub=VideoCaptionDataset(msvd_ok[:100],MSVD_VIDEO_DIR,
                                msvd_captions,NUM_FRAMES)
    r=run_evaluation_full(m,ds_sub,label=f"LLaVA({q_label})",
                           max_samples=100,compute_per_video=False)
    quant_abl[q_label]={"scores":r["scores"],"timing":r["timing"]}
    show_scores(r["scores"],f"LLaVA-NeXT ({q_label})","MSVD (100)")
    print(f"  mean latency: {r['timing']['mean_s']:.3f}s/video")
    del m; gc.collect(); torch.cuda.empty_cache() if DEVICE=="cuda" else None

with open(os.path.join(RESULTS_DIR,"LLaVA_NeXT_quantisation_ablation.json"),"w") as f:
    json.dump(quant_abl,f,indent=2)
print("Saved LLaVA_NeXT_quantisation_ablation.json")


#### Quantisation ablation chart

In [ ]:
# ── Inline quantisation ablation chart ──────────────────────────────────────
_qt_p = os.path.join(RESULTS_DIR,'LLaVA_NeXT_quantisation_ablation.json')
if os.path.exists(_qt_p):
    with open(_qt_p) as _f: _qt = json.load(_f)
    if _qt:
        _qlabels = list(_qt.keys())
        _fig, _axes = plt.subplots(1,2,figsize=(10,4))
        for _ax, _metric in zip(_axes,['CIDEr','BLEU_4']):
            _vals=[_qt[q].get('scores',_qt[q]).get(_metric,0) for q in _qlabels]
            _lat =[_qt[q].get('timing',{}).get('mean_s',0) for q in _qlabels]
            _bars=_ax.bar(range(len(_qlabels)),_vals,color=['#4C72B0','#C44E52'],alpha=0.85)
            for _b,_v in zip(_bars,_vals):
                _ax.text(_b.get_x()+_b.get_width()/2,_b.get_height()+0.002,
                         f'{_v:.4f}',ha='center',va='bottom',fontsize=9)
            _ax.set_xticks(range(len(_qlabels)))
            _ax.set_xticklabels([f'{q}\n({_lat[i]:.2f}s/vid)' for i,q in enumerate(_qlabels)],fontsize=9)
            _ax.set_title(f'LLaVA-NeXT Quantisation — {_metric}\n(MSVD 100-video subset)',fontsize=11)
            _ax.set_ylabel(_metric,fontsize=10)
            _ax.spines[['top','right']].set_visible(False)
            _ax.yaxis.grid(True,alpha=0.3,linestyle='--'); _ax.set_axisbelow(True)
        plt.tight_layout()
        _fig_path=os.path.join(RESULTS_DIR,'LLaVA_quantisation_inline.png')
        plt.savefig(_fig_path,dpi=150,bbox_inches='tight')
        plt.show()
        if MLFLOW_ACTIVE:
            log_ablation(_qt,'LLaVA-NeXT','MSVD','quantisation')
            log_figure(_fig_path)


In [ ]:
import glob
print("\n── Member C: all saved artifacts ──────────────────────")
for f in sorted(glob.glob(os.path.join(RESULTS_DIR,"LLaVA*"))+
                glob.glob(os.path.join(RESULTS_DIR,"Video_LLaMA*"))):
    print(f"  {os.path.basename(f):<55} {os.path.getsize(f):>8,} B")
